# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook guides you in loading and exploring the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya, using the `mlcroissant` library. You will load the data, review its structure, extract data using entity `@id`s, and perform exploratory analysis and visualization.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
import matplotlib.pyplot as plt
import seaborn as sns

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print("Dataset Name:", metadata.name)
print("Description:", metadata.description)
print("Published:", metadata.datePublished)
print("License:", metadata.license)
# If you want to see all keys in metadata, you can check them like so:
# print(metadata.to_json().keys())

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

Entities in the Croissant schema are referenced by their `@id` values. Let's list all record sets and their corresponding fields.

In [ ]:
# List all available record sets and their fields using @id
record_sets = dataset.metadata.recordSet
if not record_sets or len(record_sets) == 0:
    print("No record sets found in metadata. Fetching records directly.")
else:
    for rs in record_sets:
        print(f"Record Set @id: {rs['@id']} | Name: {rs.get('name','<no name>')}")
        # Show field IDs
        if 'field' in rs:
            for f in rs['field']:
                print(f"  Field @id: {f['@id']} | Name: {f.get('name','<no name>')} | Data type: {f.get('dataType','<no dtype>')}")
        print()

# As the recordSet might be empty (typically in summary metadata), let's try loading the main record set directly and preview some records.
# Find the most likely record set @id from the metadata
main_record_set_id = None
if record_sets and len(record_sets) > 0:
    main_record_set_id = record_sets[0]['@id']
else:
    # Try to infer the main record set id from typical conventions
    # For this FAIR² dataset, the main record set may have a canonical @id
    # You can load the first available records to inspect them.
    for rec in dataset.records():
        print("Sample Record:", rec)
        break


## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use record set and field `@id`s obtained above.

You can reference entities using their `@id`. Let's load the main survey record set into a DataFrame.

In [ ]:
# You should replace <record_set_id> with the actual record set @id found above.
# For demonstration, we'll attempt to extract the main record set based on either metadata or the default.

# Try to fetch available record set ids
record_sets = dataset.metadata.recordSet
df = None
record_set_ids = []

if record_sets and len(record_sets) > 0:
    # Collect all record set @ids
    for rs in record_sets:
        record_set_ids.append(rs['@id'])
else:
    # Attempt default loading
    try:
        records = list(dataset.records())
        df = pd.DataFrame(records)
        print("Loaded records (default set):")
        print(df.columns.tolist())
        print(df.head())
    except Exception as e:
        print("Unable to load records directly:", e)

dataframes = {}
for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df_rs = pd.DataFrame(records)
            dataframes[record_set_id] = df_rs
            print(f"Record set '{record_set_id}' columns:", df_rs.columns.tolist())
            print(df_rs.head())
    except Exception as e:
        print(f"Error loading records for {record_set_id}: {e}")
if not dataframes:
    print("No record sets could be loaded into DataFrames.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

Let's pick a numeric survey score, such as `GAD-7` total score, for filtering and normalization. All references use column/entity `@id`s.

In [ ]:
# Replace these with actual @id values for each entity/column
# For this dataset, suppose GAD-7 total score is stored as 'gad_7_total_score' and we group by 'village' (these should be their @id, e.g. 'https://api.app.sen.science/frontiers/7160411/gad_7_total_score')
# To explore, let's find out the actual column names (usually they match @id, or are easily mapped)

# Pick the first dataframe loaded above, if any
if dataframes:
    # Use the first available record set
    rs_id = list(dataframes.keys())[0]
    df = dataframes[rs_id]
    print("Available columns:", df.columns.tolist())
    # Find a numeric field: GAD-7, PHQ-9, PSQ scores are all likely candidates
    possible_numeric_fields = [col for col in df.columns if 'gad' in col or 'phq' in col or 'psq' in col or 'score' in col]
    print("Numeric fields candidates:", possible_numeric_fields)

    # For demonstration, let's select 'GAD_7_total_score' (replace with actual @id or column name if different)
    numeric_field_id = possible_numeric_fields[0] if possible_numeric_fields else df.columns[0]

    # Filter by threshold
    threshold = 10
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group field: use 'village' if available
    possible_group_fields = [col for col in df.columns if 'village' in col or 'education' in col or 'marital_status' in col]
    group_field_id = possible_group_fields[0] if possible_group_fields else None
    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped data by {group_field_id}:")
        print(grouped_df.head())
else:
    print("No dataframe available from loaded record sets.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Let's plot the distribution of GAD-7 scores and visualize group means for a demographic attribute (e.g. village or education).

In [ ]:
# Visualize score distributions and group means
if dataframes:
    rs_id = list(dataframes.keys())[0]
    df = dataframes[rs_id]
    possible_numeric_fields = [col for col in df.columns if 'gad' in col or 'phq' in col or 'psq' in col or 'score' in col]
    numeric_field_id = possible_numeric_fields[0] if possible_numeric_fields else df.columns[0]

    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field_id], bins=12, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    possible_group_fields = [col for col in df.columns if 'village' in col or 'education' in col or 'marital_status' in col]
    group_field_id = possible_group_fields[0] if possible_group_fields else None
    if group_field_id:
        group_data = df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        plt.figure(figsize=(10,5))
        sns.barplot(data=group_data, x=group_field_id, y=numeric_field_id)
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No dataframe available for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The FAIR² Mental Health Survey Dataset offers rich demographic and psychological survey data, referenced via Croissant `@id`s for robust data infrastructure.
- Numeric scores (e.g., GAD-7, PHQ-9) allow for filtering, normalization, and grouping by demographic attributes.
- Visualizations highlight data distributions and potential differences in mental health indicators across groups.
- For advanced analysis, always use entity `@id` to ensure schema consistency.